In [2]:
import json
from openai import OpenAI
from tqdm import tqdm
from datasets import load_dataset

/srv/nlprx-lab/share6/gramesh31/LLM-ambiguity/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from huggingface_hub import login
login(token="")

In [4]:
from collections import deque
dataset = []
cur_stems = deque([""]*5)
ds = load_dataset("PromptTensor/prompttensor-promptbank")
for item in tqdm(ds['train']):
    if item['intent'] in ['planning', 'generation']:
        if item['prompt_text'][:15] not in cur_stems:
            dataset.append(item['prompt_text'])
            cur_stems.append(item['prompt_text'][:15])
            cur_stems.popleft()


100%|██████████| 7040/7040 [00:00<00:00, 8997.02it/s]


In [5]:
len(dataset)

1371

In [ ]:
filtered = []
for p in dataset:
    if len(p.split('. ')) >= 4:
        filtered.append(p)
        print(len(p))
        print(p[:15])
        print('=='*20)

In [7]:
problems = [(i, p) for i, p in enumerate(filtered)]

In [ ]:
client = OpenAI(api_key="")

In [9]:
example1_unclear = r"""Design a six-month systematic review workflow for epigenetic trauma markers research. Choose an amount of time and split it up, where each has its own tasks, rules for ensuring high scores, and points that quantify how different each is. Include timeline milestones, team role assignments, and documentation standards for reproducibility across multidisciplinary collaborators."""
example1_clear = r"""Design a six-month systematic review workflow for epigenetic trauma markers research. Outline four distinct phases with specific deliverables, quality assessment protocols for included studies, and heterogeneity evaluation criteria. Include timeline milestones, team role assignments, and documentation standards for reproducibility across multidisciplinary collaborators."""
example4_unclear = r"""Make a guide for the duration before official registration that contains rules for testing related to solving strong negative emotion across many official locations. Specify primary and secondary endpoints, stopping rules for efficacy and futility, analysis plan including missing data handling, and data sharing commitments. Include power analysis and sensitivity analyses."""
example4_clear = r"""Develop a pre-registration protocol for a multi-site clinical trial on depression treatment. Specify primary and secondary endpoints, stopping rules for efficacy and futility, analysis plan including missing data handling, and data sharing commitments. Include power analysis and sensitivity analyses."""
example3_unclear = r"""Critique an experimental design for confound vulnerability, demand characteristics, and ecological validity threats. Identify double-blind failures, social desirability contamination, and artificial task constraints. Suggest design modifications strengthening internal and external validity. When creating the response, format the points separately and stacked, with descriptions of similar points to group them."""
example3_clear = r"""Critique an experimental design for confound vulnerability, demand characteristics, and ecological validity threats. Identify double-blind failures, social desirability contamination, and artificial task constraints. Suggest design modifications strengthening internal and external validity. Provide the response as a bullet list with clear section headers."""
example2_unclear = r"""Write about existing content on topics of organizations that distribute authority automatically, especially for official cases. Map disciplinary contributions from political science, computer science, and organizational theory. Identify definitional ambiguities, empirical evidence gaps, and normative assumptions embedded in the literature."""
example2_clear = r"""Synthesize a scoping review on decentralized autonomous organizations in governance contexts. Map disciplinary contributions from political science, computer science, and organizational theory. Identify definitional ambiguities, empirical evidence gaps, and normative assumptions embedded in the literature."""

In [10]:
clear_exs = [example1_clear, example2_clear, example3_clear, example4_clear]
unclear_exs = [example1_unclear, example2_unclear, example3_unclear, example4_unclear]

examples = []
for clr, unclr in zip(clear_exs, unclear_exs):
    d = {}
    for cs, us in zip(clr.split('.'), unclr.split('.')):
        if cs != us:
            d[cs] = us

    example = f"""Original Prompt: {example1_clear}
Output: {d}"""

    examples.append(example)

example_string = ""
for i in range(len(examples)):
    example_string += f"**Example {i}:**\n" + examples[i] + "\n\n"
print(example_string)

**Example 0:**
Original Prompt: Design a six-month systematic review workflow for epigenetic trauma markers research. Outline four distinct phases with specific deliverables, quality assessment protocols for included studies, and heterogeneity evaluation criteria. Include timeline milestones, team role assignments, and documentation standards for reproducibility across multidisciplinary collaborators.
Output: {' Outline four distinct phases with specific deliverables, quality assessment protocols for included studies, and heterogeneity evaluation criteria': ' Choose an amount of time and split it up, where each has its own tasks, rules for ensuring high scores, and points that quantify how different each is'}

**Example 1:**
Original Prompt: Design a six-month systematic review workflow for epigenetic trauma markers research. Outline four distinct phases with specific deliverables, quality assessment protocols for included studies, and heterogeneity evaluation criteria. Include timelin

In [ ]:
USER_TEMPLATE = """You are given a writing question prompt consisting of multiple sentences.

Your task has two steps:

1. Identify the sentence in the prompt that is most critical to understanding what the writing question is asking. This should be the sentence whose removal or corruption would most severely impair comprehension of the task.

2. For this sentence, produce an **obscured version** that changes the underlying meaning such that, if substituted back into the prompt, the overall meaning of the problem would become unclear. You must obscure the sentence enough that the writing task is no longer understandable logically if substituted in. An LLM should not respond to the modified prompt because of how unclear it is.

Note: Do not only create ambiguity through scope ambiguous vocabulary/quantifiers (words like "maybe", "sometimes", "vaguely" or "random") or words synonymous with 'vague' (words like "mystery", "random", "something", "however", "any", "may", "specified") for the obfuscation. Instead, consider using techniques like deleting small portions of key information and replacing words with less-specific synonyms. Just make sure the meaning is obfuscated enough that the whole prompt becomes unclear and unanswerable.

#####

Output Instructions:

You must output **only** the following JSON-like object, with no extra text before or after:

{{
   **original sentence**: **obscured version**
}}

Important constraints:
- Choose only 1 sentence to obfuscate that would most impair comprehension of the task if changed.
- The obscured version should not be longer than the original sentence.
- The key must be an exact sentence copied verbatim from the prompt.
- The value must be an changed version of only that sentence.
- Do not paraphrase the original text in the keys.
- Do not include explanations, bullet points, or formatting outside the schema.

Here is the writing question prompt:
"{prompt}"


Here are examples of good obfuscations:

{example_string}

NOTES:
- The changed sentence still has a similar meaning as the original, but is more vague.
- The changed sentence would make the overall prompt unclear if substituted in.
- Every part of the changed sentence is more vague than the original, so the whole new sentence is ambiguous.
- The changed sentence should be unclear on its own.

#####
"""

In [12]:
SYSTEM_PROMPT = "You are a research assistant working on a project investigating how large language models (LLMs) process ambiguity in prompts they are given. The prompts being evaluated are complex, therefore you are trying to see what area of the prompt causes the underlying task asked in the prompt to be unclear to the LLM. You are currently working on producing a dataset for the project by generating ambiguous versions of given prompts."

In [61]:
def generate_unclear_examples(prompt_text):
    response = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[
                {"role": "system", "content": ""},
                {"role": "user", "content": USER_TEMPLATE.format(prompt=prompt_text, example_string=example_string)},
            ],
            temperature=0.7,
            max_completion_tokens=500,
        )
    content = response.choices[0].message.content.strip()
    return content

In [38]:
print(problems[0][1])

Write a concise summary of a hypothetical research paper on AI ethics in healthcare that would be suitable for a CTO audience. Structure your response with three distinct sections: Business Implications (one paragraph on risk mitigation and opportunity identification), Technical Implementation Considerations (bullet points on model validation requirements and data governance needs), and Strategic Recommendations (numbered list of three actionable next steps with estimated resource requirements). Exclude all philosophical ethics discussions. Focus on operationalizable insights with concrete examples from diagnostic imaging applications. Format the output as a memo with a brief header (To/From/Subject/Date). Constraints: Keep it under 350 words; Use clear section headings; Include exactly three key takeaways.


In [43]:
prompt = problems[0][1]
unclear_prompt = generate_unclear_examples(prompt)


In [44]:
print(unclear_prompt)

{"Structure your response with three distinct sections: Business Implications (one paragraph on risk mitigation and opportunity identification), Technical Implementation Considerations (bullet points on model validation requirements and data governance needs), and Strategic Recommendations (numbered list of three actionable next steps with estimated resource requirements).":"Arrange the response into three parts: Business Effects, Technical Notes, and Strategy Suggestions, each with the listed details and approximate staffing needs."}


In [45]:
for i, (id,prompt) in enumerate(tqdm(problems)):
    unclear_data = generate_unclear_examples(prompt)
    if "```json" in unclear_data:
        unclear_data = unclear_data.split("```json")[1].split("```")[0].strip()

    with open(f'writing/{id}.txt', 'w', encoding='utf-8') as f:
        f.write(unclear_data)

100%|██████████| 385/385 [09:49<00:00,  1.53s/it]


In [78]:
failed_ids = []
# for each output in math/{id}.txt check to see that the original sentence is actually in the prompt, and that it is only 1 sentence
for id in range(len(problems)):
    prompt = problems[id][1]
    # check if file exists
    try:
        with open(f'writing/{id}.txt', 'r') as f:
            content = f.read().strip()
    except:
        failed_ids.append(id)
        continue
    # read the output file    
    with open(f'writing/{id}.txt', 'r') as f:
        content = f.read().strip()
    if content[-2:] == ',}':
        content = content[:-2] + content[-1:]
        with open(f'writing/{id}.txt', 'w', encoding='utf-8') as f:
            f.write(content)
    if content[-3:] == ',\n}':
        content = content[:-3] + content[-2:]
        with open(f'writing/{id}.txt', 'w', encoding='utf-8') as f:
            f.write(content)
    # parse the content as a dictionary
    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        print(f"Output for problem {id} is not valid JSON.")
        failed_ids.append(id)
        continue
    # check that there is only 1 key-value pair
    if len(data) != 1:
        print(f"Output for problem {id} does not contain exactly 1 key-value pair.")
        failed_ids.append(id)
        continue
    # check that the key is a contiguous portion of the original prompt
    key = list(data.keys())[0]
    if key not in prompt:
        print(f"Key '{key}' in output for problem {id} is not a contiguous portion of the original prompt.")
        failed_ids.append(id)
        continue
    # check that the key is only 1 sentence    
    if len(key.split('. ')) >= 2:
        print(f"Key '{key}' in output for problem {id} is not only one sentence.")
        failed_ids.append(id)
        continue

print(len(failed_ids))

0


In [76]:
for i, id in enumerate(tqdm(failed_ids)):
    id, prompt = problems[id]
    unclear_data = generate_unclear_examples(prompt)
    if "```json" in unclear_data:
        unclear_data = unclear_data.split("```json")[1].split("```")[0].strip()

    with open(f'writing/{id}.txt', 'w', encoding='utf-8') as f:
        f.write(unclear_data)

100%|██████████| 1/1 [00:01<00:00,  1.67s/it]
